# MIMYR：Zhuang-ABCA-2 论文留出重建 + 自定义三折重建

**只加载已有权重，不训练、不微调、不改模型头或词表。** 文件名保留为
`merfish_pretrained_3fold.ipynb`，内容已切换到下载好的真实 CCF 数据。

顺序运行两组实验，结果独立保留：

1. `paper_rq3`：按当前作者代码的 4/1/5 划分重建五张固定测试切片。
2. `custom_3fold`：沿用之前“五张连续切片、依次留出内部三张”的规则，默认使用 Zhuang 的
   `sec_031、032、033、034、035`，三个目标为 `sec_032、033、034`。

两组默认使用相同的源模型权重，对应 **zero-shot**。如需作者已微调版本，可在配置处选择
`WEIGHT_SETTING='author_finetuned'`，直接加载已下载的 `best_model_rq3.pt` 与
`Zhuang-2_epoch60_model.pt`，同样不会训练。两个权重设定的结果目录也不同。

两组均使用当前旧版仓库 `rq3` 的三阶段模型推理、70 步扩散、2.5 倍位置候选采样、FP32、
原表达生成批量 1500、500 个新 token、top-k 5 和 KV cache。不会自动降低批量或精度。
这里复现的是本地旧版实现和已有权重，不混用作者新版 API 的默认设置。

与原代码一致，重建获得目标切片的预处理后细胞数、空间边界和切面位置。
生成阶段不会把目标表达或真实细胞类别当作条件。该设置包含目标几何信息；与完全不使用目标几何的
盲重建比较时应注明。三折是三次冻结模型留出预测，不是训练三套模型。

## 1. 数据和论文划分

使用下载 notebook 已生成的数据：

```text
MIMYR/data/
├── ccf-ABCA-2.csv
└── Zhuang-ABCA-2/sec_*.h5ad
```

真实 CCF 坐标按完整字符串 `cell_label` 对齐 `obs_names`，过滤无有效 CCF 的细胞。
与作者代码一致，内部坐标顺序为 `[z_ccf, y_ccf, x_ccf]`，单位 mm。

作者先排序全部 66 个文件，去掉首尾各两张，再使用零起始索引；**不能先删掉无 CCF 的整张切片再编号**。

| 用途 | 切片编号 |
|---|---|
| 原微调训练集（本次不训练） | 018、025、044、049 |
| 原验证集（本次不调参） | 034 |
| 固定测试集 | 021、028、037、047、051 |

| 重建目标 | 作者指定参考 |
|---|---|
| sec_021 | sec_018、sec_025 |
| sec_028 | sec_025、sec_034 |
| sec_037 | sec_034、sec_044 |
| sec_047 | sec_044、sec_049 |
| sec_051 | sec_049、sec_049 |

最后一对重复参考保留作者原实现。验证切片也可能作为重建参考。
微调设定在论文中为 4/1/5；zero-shot 使用源模型直接测试，不在这四张上训练。
划分来自本地 `data_loader.py` 的 `rq3` 分支和已下载文件；
[论文](https://doi.org/10.1101/2025.11.24.690239)、[作者数据](https://huggingface.co/datasets/gkrieg13/mimyr-data)。

In [ ]:
import os
import sys
import uuid
from datetime import datetime, timedelta, timezone
from pathlib import Path

GPU_ID = '0'  # 更改物理 GPU 后重启内核
if 'torch' in sys.modules:
    raise RuntimeError('请重启内核，从本单元开始运行，以正确选择 GPU。')
if GPU_ID is not None:
    os.environ['CUDA_VISIBLE_DEVICES'] = GPU_ID
PROJECT = next((p.resolve() for p in [Path.cwd(), *Path.cwd().parents]
                if (p / 'main.py').is_file() and (p / 'models/combined_model.py').is_file()), None)
if PROJECT is None:
    raise FileNotFoundError('请从 MIMYR 或 notebooks 目录打开本 notebook。')
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

DATA_DIR = PROJECT / 'data'
SECTION_DIR = DATA_DIR / 'Zhuang-ABCA-2'
CCF_CSV = DATA_DIR / 'ccf-ABCA-2.csv'
METADATA_DIR = PROJECT / 'models/generative_transformer/metadata'
WEIGHT_SETTING = 'zero_shot'       # zero_shot / author_finetuned
FIVE_SECTIONS = [31, 32, 33, 34, 35]
SEED = 42
RUN_ID = None                     # 恢复已有运行时填其原 RUN_ID，并将 RESUME=True
RESUME = False
if RUN_ID is None:
    RUN_ID = datetime.now(timezone(timedelta(hours=8))).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:8]
if Path(RUN_ID).name != RUN_ID or RUN_ID in {'.', '..'} or any(c in RUN_ID for c in '/\\:'):
    raise ValueError('RUN_ID 必须为单个目录名。')
RESULTS_ROOT = PROJECT / 'results/zhuang_abca2_reconstruction' / WEIGHT_SETTING
print('数据:', SECTION_DIR)
print('权重设定:', WEIGHT_SETTING, '| RUN_ID:', RUN_ID, '| 恢复:', RESUME)
print('论文输出:', RESULTS_ROOT / 'paper_rq3' / RUN_ID)
print('三折输出:', RESULTS_ROOT / 'custom_3fold' / RUN_ID)

In [ ]:
import ast
import copy
import gc
import hashlib
import json
import pickle
import platform
import time
import urllib.request
from types import SimpleNamespace

import anndata as ad
import numpy as np
import pandas as pd
import torch
from IPython.display import display

weight_names = {
    'zero_shot': ('best_model_rq1.pt', 'TG-base4_epoch4_model.pt'),
    'author_finetuned': ('best_model_rq3.pt', 'Zhuang-2_epoch60_model.pt'),
}
if WEIGHT_SETTING not in weight_names:
    raise ValueError('未知权重设定。')
cluster_name, expression_name = weight_names[WEIGHT_SETTING]
CHECKPOINTS = {
    'location': PROJECT / 'model_checkpoints/smoothtune_conditional_ddpm_2d_checkpoint_400.pt',
    'cluster': PROJECT / 'model_checkpoints' / cluster_name,
    'expression': PROJECT / 'model_checkpoints' / expression_name,
}
CFG = {
    'run_mode': 'inference', 'data_mode': 'rq3', 'data_label': 'cluster',
    'location_inference_type': 'model', 'cluster_inference_type': 'model',
    'expression_inference_type': 'model', 'kde_bandwidth': 0.01,
    'guidance_signal': 0.01, 'batch_size': 1024, 'device': 'cuda',
    'metrics': ['soft_spearman_correlation', 'soft_f1'], 'metric_sampling': 100,
    'full_gene_panel': True, 'meta_info': '4hierarchy_metainfo_mouse_geneunion2_DAG.pt',
    'location_model_checkpoint': str(CHECKPOINTS['location']),
    'cluster_model_checkpoint': str(CHECKPOINTS['cluster']),
    'expression_model_checkpoint': str(CHECKPOINTS['expression']),
}

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(4 * 1024**2), b''):
            h.update(chunk)
    return h.hexdigest()

def git_blob_sha(data):
    return hashlib.sha1(b'blob ' + str(len(data)).encode() + b'\0' + data).hexdigest()

def write_json_new(path, value):
    # 所有结果均独占创建；已有文件绝不覆盖。
    with Path(path).open('x', encoding='utf-8') as stream:
        json.dump(value, stream, ensure_ascii=False, indent=2, allow_nan=False)

def write_csv_new(path, table, index=False):
    with Path(path).open('x', encoding='utf-8', newline='') as stream:
        table.to_csv(stream, index=index)

def paper_split_from_source(project):
    tree = ast.parse((Path(project) / 'data_loader.py').read_text(encoding='utf-8-sig'))
    matches = [node for node in ast.walk(tree) if isinstance(node, ast.If)
               and isinstance(node.test, ast.Compare)
               and isinstance(node.test.left, ast.Attribute) and node.test.left.attr == 'mode'
               and len(node.test.comparators) == 1
               and isinstance(node.test.comparators[0], ast.Constant)
               and node.test.comparators[0].value == 'rq3']
    if len(matches) != 1:
        raise ValueError('无法唯一定位作者 rq3 划分。')
    wanted = {'train_indices', 'val_indices', 'test_indices', 'ref_indices'}
    split = {node.targets[0].id: ast.literal_eval(node.value) for node in matches[0].body
             if isinstance(node, ast.Assign) and isinstance(node.targets[0], ast.Name)
             and node.targets[0].id in wanted}
    expected = {'train_indices': [15, 21, 37, 42], 'val_indices': [29],
                'test_indices': [18, 24, 32, 40, 44],
                'ref_indices': [15, 21, 21, 29, 29, 37, 37, 42, 42, 42]}
    if split != expected:
        raise ValueError('作者代码划分发生变化，请核对后再运行；未自动替换划分。')
    return split

def build_protocols(section_dir, split, five_sections):
    all_paths = sorted(Path(section_dir).glob('*.h5ad'), key=lambda p: p.name)
    if len(all_paths) != 66:
        raise ValueError(f'要求完整的 ABCA-2：66 个文件，实际 {len(all_paths)}。')
    ordered = all_paths[2:-2]  # 必须在过滤空 CCF 切片之前编号，保持作者索引。
    original = {key: [ordered[i].name for i in indices] for key, indices in split.items()}
    paper = [{'case_id': f'test_{i + 1:02d}_{target.stem}', 'target': target.name,
              'reference': [ordered[j].name for j in split['ref_indices'][2*i:2*i+2]],
              'heldout_index': split['test_indices'][i]} for i, target in
             enumerate(ordered[j] for j in split['test_indices'])]
    if len(five_sections) != 5 or len(set(five_sections)) != 5 or five_sections != sorted(five_sections):
        raise ValueError('FIVE_SECTIONS 必须是按切片顺序排列的五个不同编号。')
    five = [f'sec_{i:03}.h5ad' for i in five_sections]
    if any(not (Path(section_dir) / name).is_file() for name in five):
        raise FileNotFoundError('自定义五张切片不存在。')
    custom = [{'case_id': f'fold_{i}_{Path(five[i]).stem}', 'target': five[i],
               'reference': [five[i-1], five[i+1]], 'heldout_index': i,
               'visible': [name for j, name in enumerate(five) if j != i]} for i in [1, 2, 3]]
    protocols = {'paper_rq3': {'original_split': original, 'cases': paper, 'rng_policy': 'paper_sequential'},
                 'custom_3fold': {'five_sections': five, 'cases': custom, 'rng_policy': 'seed_each_fold'}}
    for name, protocol in protocols.items():
        for case in protocol['cases']:
            if case['target'] in case['reference'] or len(case['reference']) != 2:
                raise ValueError(f'{name}: 参考中包含测试目标，或参考数量不是两张。')
    return protocols

SPLIT = paper_split_from_source(PROJECT)
PROTOCOLS = build_protocols(SECTION_DIR, SPLIT, FIVE_SECTIONS)
for name, protocol in PROTOCOLS.items():
    print('\n', name, '|', protocol['rng_policy'])
    display(pd.DataFrame(protocol['cases']))
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

## 2. 原始元数据和已有权重

核对原来的 5274 类、2000 基因词表及层级映射。不重新定义类别、不重新训练。
缺少 `hierarchy.pkl` 时只下载作者固定版本的该元数据文件；已有模型权重不下载、不改写。

In [ ]:
missing_weights = [str(p) for p in CHECKPOINTS.values() if not p.is_file()]
if missing_weights:
    raise FileNotFoundError('先把已有权重复制到服务器对应目录：\n' + '\n'.join(missing_weights))

META_PATH = METADATA_DIR / CFG['meta_info']
ID_PATH = PROJECT / 'id_to_subclass.pkl'
hierarchy_path = METADATA_DIR / 'hierarchy.pkl'
expected_blobs = {
    META_PATH: '0c0b63d501c42f4a2febf65ddf5a1c858f0ca99f',
    ID_PATH: '266bf5fde2b6c89265a0b2a638c42c06cab65ecf',
    hierarchy_path: 'c54536a034ae8c6d2b03f14866aacc22232e1921',
}
for path in [META_PATH, ID_PATH]:
    if not path.is_file() or git_blob_sha(path.read_bytes()) != expected_blobs[path]:
        raise ValueError(f'缺少或不匹配的原始元数据：{path}')
if not hierarchy_path.exists():
    url = ('https://raw.githubusercontent.com/gkrieg/mimyr/'
           'f96115d978ed619add1bb65b21a85b05ba41c13f/'
           'src/mimyr/models/generative_transformer/metadata/hierarchy.pkl')
    with urllib.request.urlopen(url, timeout=60) as response:
        content = response.read()
    if git_blob_sha(content) != expected_blobs[hierarchy_path]:
        raise ValueError('下载的 hierarchy.pkl 校验失败。')
    hierarchy_path.write_bytes(content)
if git_blob_sha(hierarchy_path.read_bytes()) != expected_blobs[hierarchy_path]:
    raise ValueError('hierarchy.pkl 与固定作者版本不匹配。')

META = torch.load(META_PATH, map_location='cpu', weights_only=False)
ID_TO_LABEL = pickle.loads(ID_PATH.read_bytes())
HIERARCHY = pickle.loads(hierarchy_path.read_bytes())
if len(ID_TO_LABEL) != 5274 or set(ID_TO_LABEL) != set(range(5274)):
    raise ValueError('预训练细胞类别映射不完整。')
for label in ID_TO_LABEL.values():
    if ('cluster', label) not in HIERARCHY:
        raise ValueError(f'缺少预测类别的层级映射：{label}')
EDGES = {c: pickle.loads((METADATA_DIR / f'edges_{c}.pkl').read_bytes()) for c in 'xyz'}
print('模型词表:', len(META['token_set']), '| 模型基因:', len(META['gene_set']),
      '| 预训练类别:', len(ID_TO_LABEL))

## 3. 读取真实 CCF，审查两组使用的全部切片

CCF 按字符串 cell ID 合并，保留有有限坐标的细胞。只读切片元数据进行审查，暂不加载全部表达矩阵。
这里不额外删除、重新排序整张切片。审查表中的细胞数尚未应用原表达预处理的过滤。

In [ ]:
if not CCF_CSV.is_file():
    raise FileNotFoundError(f'缺少真实 CCF 表：{CCF_CSV}')
CCF = pd.read_csv(CCF_CSV, dtype={'cell_label': str},
                  usecols=['cell_label', 'x', 'y', 'z']).set_index('cell_label')
if not CCF.index.is_unique:
    raise ValueError('CCF 表的 cell_label 重复。')
LABEL_TO_ID = {label: int(token) for token, label in ID_TO_LABEL.items()}

def ccf_match(obs_names, ccf):
    names = pd.Index(obs_names).astype(str)
    if not names.is_unique or not ccf.index.is_unique:
        raise ValueError('细胞 ID 必须唯一。')
    coords = ccf.reindex(names)[['x', 'y', 'z']].to_numpy(dtype=float)
    return coords, np.isfinite(coords).all(axis=1)

def input_sections(protocol):
    names = {case['target'] for case in protocol['cases']}
    names.update(name for case in protocol['cases'] for name in case['reference'])
    for name in protocol.get('original_split', {}).get('train_indices', []):
        names.add(name)
    for name in protocol.get('original_split', {}).get('val_indices', []):
        names.add(name)
    return sorted(names)

audit_rows = []
for name in sorted(set().union(*(set(input_sections(p)) for p in PROTOCOLS.values()))):
    a = ad.read_h5ad(SECTION_DIR / name, backed='r')
    try:
        coords, valid = ccf_match(a.obs_names, CCF)
        required = {'class', 'subclass', 'supertype', 'cluster'}
        if not required.issubset(a.obs.columns):
            raise ValueError(f'{name} 缺少原模型使用的层级标签。')
        unknown = sorted(set(a.obs.loc[valid, 'cluster'].dropna().astype(str)) - set(LABEL_TO_ID))
        if unknown:
            raise ValueError(f'{name} 的 cluster 不在固定词表中：{unknown[:5]}')
        if not valid.any():
            raise ValueError(f'{name} 没有可用 CCF 细胞。')
        if not a.obs_names.is_unique or not a.var_names.is_unique:
            raise ValueError(f'{name} 的 cell/gene ID 不唯一。')
        overlap = len(set(a.var_names).intersection(META['gene_set']))
        if not overlap:
            raise ValueError(f'{name} 与模型没有共同基因。')
        audit_rows.append({'section': name, 'n_input': a.n_obs, 'n_ccf': int(valid.sum()),
                           'n_without_ccf': int((~valid).sum()), 'measured_genes': a.n_vars,
                           'model_overlap_genes': overlap, 'x_ccf_mean_mm': float(coords[valid, 0].mean())})
    finally:
        a.file.close()
AUDIT = pd.DataFrame(audit_rows)
display(AUDIT)
print('输入检查完成；此单元没有产生重建结果，也没有创建实验输出目录。')

## 4. 加载模型与原始数据预处理

保留旧版 `rq3` 的原始调用。数据预处理直接调用原 `harmonize_dataset`：按其条件 log1p、
过滤细胞、添加模型基因及坐标分箱，沿用该函数默认的元数据设置，不替换为新版默认值。
生成阶段只接收目标几何及原技术元数据，不接收目标真实类别或表达作为条件。

原表达批量 1500 的显存需求较大；遇到 OOM 会保留失败记录并停止，不自动改变实验配置。
本 notebook 没有训练调用。论文组按原入口顺序推进随机数；自定义三折每折重新设 seed 42。

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('需要可用 CUDA 才运行原始推理配置。请在服务器 GPU 内核中执行后续单元。')

import __main__
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from main import TrainConfig as PaperTrainConfig
from models.combined_model import CombinedModel
from models.generative_transformer.data_util import harmonize_dataset
from inference import Inferernce
from evaluator import Evaluator
from models.biological_model import KDEModelForGuidance

__main__.TrainConfig = PaperTrainConfig  # 旧检查点的反序列化别名，不修改参数

def seed_all(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def rng_state():
    return {'numpy': np.random.get_state(), 'torch': torch.get_rng_state(),
            'cuda': torch.cuda.get_rng_state_all()}

def restore_rng(state):
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch'])
    torch.cuda.set_rng_state_all(state['cuda'])

def source_settings_check():
    tree = ast.parse((PROJECT / 'inference.py').read_text(encoding='utf-8-sig'))
    calls = [node for node in ast.walk(tree) if isinstance(node, ast.Call)
             and isinstance(node.func, ast.Attribute) and node.func.attr == 'generate_cell_genesis']
    if len(calls) != 1:
        raise ValueError('表达生成入口发生变化。')
    params = {k.arg: ast.literal_eval(k.value) for k in calls[0].keywords
              if k.arg in {'batch_size', 'max_new_tokens', 'top_k', 'fast'}}
    expected = {'max_new_tokens': 500, 'top_k': 5, 'batch_size': 1500, 'fast': True}
    if params != expected or PaperTrainConfig().n_timesteps != 70:
        raise ValueError('推理参数与原设置不一致，未自动修改。')
    return params

ORIGINAL_GENERATION = source_settings_check()
seed_all(SEED)
model = CombinedModel(location_model_checkpoint=CFG['location_model_checkpoint'],
                      celltype_model_checkpoint=CFG['cluster_model_checkpoint'],
                      gene_exp_model_checkpoint=CFG['expression_model_checkpoint'])
for network in (model.trainer.model, model.celltype_model):
    network.eval()
    for parameter in network.parameters():
        parameter.requires_grad_(False)
evaluator = Evaluator(CFG, metadata_dir=str(METADATA_DIR) + os.sep)
PAPER_INITIAL_RNG = rng_state()

def prepared_section(name):
    a = ad.read_h5ad(SECTION_DIR / name)
    coords, valid = ccf_match(a.obs_names, CCF)
    a = a[valid].copy()
    for j, c in enumerate('xyz'):
        a.obs[f'{c}_ccf'] = coords[valid, j]
    a.obsm['aligned_spatial'] = coords[valid][:, [2, 1, 0]]
    labels = a.obs['cluster'].astype(str)
    if not labels.isin(LABEL_TO_ID).all():
        raise ValueError(f'{name} 含未知 cluster，未用多数类替代。')
    a.obs['token'] = labels.map(LABEL_TO_ID).astype(int)
    measured = a.var_names.tolist()
    a = harmonize_dataset(a, META, [str(METADATA_DIR / f'edges_{c}.pkl') for c in 'xyz'])
    if not a.n_obs:
        raise ValueError(f'{name} 原预处理后为空。')
    a.uns['measured_gene_names'] = measured
    return a

def geometry_only(a):
    columns = [c for c in ['organ', 'technology', 'species', 'disease_state'] if c in a.obs]
    g = ad.AnnData(obs=a.obs[columns].copy())
    g.obsm['aligned_spatial'] = a.obsm['aligned_spatial'].copy()
    return g

print('已加载冻结模型，未执行训练；原表达参数:', ORIGINAL_GENERATION)
print('GPU:', torch.cuda.get_device_name(0), '| 显存 GiB:', torch.cuda.get_device_properties(0).total_memory / 2**30)

## 5. 独立运行目录、结果保护与恢复

每次从头运行默认生成新的 `RUN_ID`，论文组、三折组及不同权重设定分别存储。
之前的 `results/merfish_pretrained_3fold/` 完全不动。

需要恢复时，将配置单元的 `RUN_ID` 改为原值、`RESUME=True`。配置、输入、源码、权重与环境必须一致。
完成的目标校验后复用；失败或中断的目标创建新的 `attempt_###`，不覆盖上次的预测或失败记录。
论文顺序随机数状态随每个成功目标保存，恢复时继续原序列。不要同时用两个进程恢复同一个 RUN_ID。

In [ ]:
def initialize_run(folder, identity, resume=False):
    folder = Path(folder)
    stamp = folder / 'experiment.json'
    if folder.exists():
        if not resume or not stamp.is_file() or json.loads(stamp.read_text(encoding='utf-8')) != identity:
            raise FileExistsError(f'已有结果被保留，请换 RUN_ID，或仅恢复完全相同的运行：{folder}')
    else:
        folder.mkdir(parents=True, exist_ok=False)
        write_json_new(stamp, identity)
    return folder

def new_attempt(case_dir):
    case_dir = Path(case_dir)
    case_dir.mkdir(parents=True, exist_ok=True)
    for i in range(1, 100000):
        target = case_dir / f'attempt_{i:03d}'
        try:
            target.mkdir(exist_ok=False)
            return target
        except FileExistsError:
            continue
    raise RuntimeError('尝试目录过多，请使用新 RUN_ID。')

def verify_completed_case(case_dir):
    case_dir = Path(case_dir).resolve()
    done = json.loads((case_dir / 'completed.json').read_text(encoding='utf-8'))
    attempt = (case_dir / done['attempt']).resolve()
    if not attempt.is_relative_to(case_dir):
        raise ValueError('完成记录的路径无效。')
    for name, digest in done['files'].items():
        file = (attempt / name).resolve()
        if not file.is_relative_to(attempt) or not file.is_file() or sha256(file) != digest:
            raise ValueError(f'已有结果被修改或缺失，未覆盖：{file}')
    return attempt

def make_identity(protocol_name):
    protocol = PROTOCOLS[protocol_name]
    source_paths = [PROJECT / name for name in
                    ['main.py', 'inference.py', 'evaluator.py', 'metrics.py', 'analysis.py', 'data_loader.py', 'gene_exp_model.py']]
    source_paths += sorted((PROJECT / 'models').rglob('*.py'))
    metadata_paths = [META_PATH, ID_PATH, hierarchy_path, *[METADATA_DIR / f'edges_{c}.pkl' for c in 'xyz']]
    notebook = json.loads((PROJECT / 'notebooks/merfish_pretrained_3fold.ipynb').read_text(encoding='utf-8'))
    notebook_code = '\n'.join(''.join(c['source']) for c in notebook['cells'] if c['cell_type'] == 'code')
    return {'config': CFG, 'protocol_name': protocol_name, 'protocol': protocol,
            'weight_setting': WEIGHT_SETTING, 'seed': SEED, 'generation': ORIGINAL_GENERATION,
            'input_sha256': {name: sha256(SECTION_DIR / name) for name in input_sections(protocol)},
            'ccf_sha256': sha256(CCF_CSV),
            'checkpoint_sha256': {k: sha256(v) for k, v in CHECKPOINTS.items()},
            'metadata_sha256': {str(p.relative_to(PROJECT)): sha256(p) for p in metadata_paths},
            'source_sha256': {str(p.relative_to(PROJECT)): sha256(p) for p in source_paths},
            'notebook_code_sha256': hashlib.sha256(notebook_code.encode('utf-8')).hexdigest(),
            'environment': {'python': platform.python_version(), 'torch': torch.__version__,
                            'numpy': np.__version__, 'cuda': torch.version.cuda,
                            'gpu': torch.cuda.get_device_name(0)}}

def predict_case(protocol_name, case, run_dir):
    case_dir = run_dir / case['case_id'] / f'seed_{SEED}'
    if (case_dir / 'completed.json').is_file():
        attempt = verify_completed_case(case_dir)
        restore_rng(torch.load(attempt / 'rng_after.pt', map_location='cpu', weights_only=False))
        print('复用已校验结果:', attempt, flush=True)
        return json.loads((attempt / 'metrics.json').read_text(encoding='utf-8'))
    if PROTOCOLS[protocol_name]['rng_policy'] == 'seed_each_fold':
        seed_all(SEED)
    attempt = new_attempt(case_dir)
    write_json_new(attempt / 'config.json', {**CFG, 'protocol_name': protocol_name,
                   'case': case, 'weight_setting': WEIGHT_SETTING, 'seed': SEED})
    started = time.perf_counter()
    torch.cuda.reset_peak_memory_stats()
    try:
        truth = prepared_section(case['target'])
        target_geometry = geometry_only(truth)
        references = []
        for name in case['reference']:
            a = prepared_section(name)
            references.append(geometry_only(a))
            del a
        dists = [(r.obsm['aligned_spatial'][:, -1].mean()
                  - target_geometry.obsm['aligned_spatial'][:, -1].mean()) ** 2 for r in references]
        kde_reference = references[np.argsort(dists)[1]].copy()
        kde_reference.obsm['aligned_spatial'] = kde_reference.obsm['aligned_spatial'][:, :2]
        kde = KDEModelForGuidance([kde_reference], bandwidth=CFG['kde_bandwidth'])
        kde.fit()
        token_map = SimpleNamespace(get_label_from_token=lambda tokens: [ID_TO_LABEL[int(t)] for t in tokens])
        loader = SimpleNamespace(metadata_dir=str(METADATA_DIR) + os.sep, gene_exp_model=token_map,
                                 test_slices=[target_geometry], reference_slices=references, train_slices=references)
        cfg = copy.deepcopy(CFG)
        cfg.update(artifact_dir=str(attempt), slice_index=case['heldout_index'])
        inference = Inferernce(model, kde, loader, cfg)
        with torch.no_grad():
            prediction = ad.AnnData(obs=target_geometry.obs.copy())
            prediction = inference.infer_location(prediction, target_geometry)
            prediction = inference.infer_cluster(prediction, target_geometry)
            prediction, generated_tokens = inference.infer_expression(prediction, target_geometry)
        del generated_tokens
        prediction.obs['cluster'] = [ID_TO_LABEL[int(t)] for t in prediction.obs['token']]
        prediction.obs['source_target_section'] = case['target']
        coords = np.asarray(prediction.obsm['spatial'])
        for j, c in enumerate(['z', 'y', 'x']):
            prediction.obs[f'{c}_ccf'] = coords[:, j]
        prediction.obsm['spatial_ccf_mm'] = coords[:, [2, 1, 0]].copy()  # 明确使用实际 CCF [x, y, z]
        prediction.uns['protocol_json'] = json.dumps({'name': protocol_name, 'case': case,
                    'weight_setting': WEIGHT_SETTING, 'training': False, 'target_geometry_available': True})
        prediction_path = attempt / 'prediction.h5ad'
        if prediction_path.exists():
            raise FileExistsError(prediction_path)
        prediction.write_h5ad(prediction_path, compression='gzip')
        measured = [g for g in truth.uns['measured_gene_names'] if g in set(META['gene_set'])]
        write_csv_new(attempt / 'evaluated_gene_panel.csv', pd.DataFrame({'gene': measured}))
        # 原 Evaluator 会原地降为二维；保留磁盘上的三维预测不变。
        result = evaluator.evaluate(prediction.copy(), truth, sample=CFG['metric_sampling'])
        metrics = {k: float(v) for k, v in result.items()}
        if not metrics or not all(np.isfinite(list(metrics.values()))):
            raise ValueError('指标为空或含非有限值，该目标不标记为完成。')
        metrics.update(protocol=protocol_name, case_id=case['case_id'], target=case['target'],
                       references=case['reference'], seed=SEED, weight_setting=WEIGHT_SETTING,
                       n_predicted=prediction.n_obs, n_evaluated=truth.n_obs,
                       n_measured_overlap_genes=len(measured), elapsed_seconds=time.perf_counter() - started,
                       peak_allocated_gib=torch.cuda.max_memory_allocated() / 2**30)
        write_json_new(attempt / 'metrics.json', metrics)
        with (attempt / 'rng_after.pt').open('xb') as stream:
            torch.save(rng_state(), stream)
        files = ['prediction.h5ad', 'metrics.json', 'config.json', 'evaluated_gene_panel.csv', 'rng_after.pt']
        write_json_new(case_dir / 'completed.json', {'attempt': attempt.name,
                       'files': {name: sha256(attempt / name) for name in files}})
        return metrics
    except Exception as exc:
        write_json_new(attempt / 'failed.json', {'case': case, 'error': repr(exc),
                       'note': '本次失败产物保留，未降低批量、精度或生成长度。'})
        raise
    finally:
        plt.close('all')
        gc.collect()
        torch.cuda.empty_cache()

def run_protocol(protocol_name):
    identity = make_identity(protocol_name)
    run_dir = initialize_run(RESULTS_ROOT / protocol_name / RUN_ID, identity, RESUME)
    if (run_dir / 'run_completed.json').is_file():
        done = json.loads((run_dir / 'run_completed.json').read_text(encoding='utf-8'))
        for name, digest in done['files'].items():
            if sha256(run_dir / name) != digest:
                raise ValueError(f'已完成汇总被修改：{run_dir / name}')
        for case in PROTOCOLS[protocol_name]['cases']:
            verify_completed_case(run_dir / case['case_id'] / f'seed_{SEED}')
        return pd.read_csv(run_dir / 'per_case.csv'), pd.read_csv(run_dir / 'summary.csv'), run_dir
    if not (run_dir / 'input_audit.csv').exists():
        selected = AUDIT[AUDIT['section'].isin(input_sections(PROTOCOLS[protocol_name]))]
        write_csv_new(run_dir / 'input_audit.csv', selected)
    if PROTOCOLS[protocol_name]['rng_policy'] == 'paper_sequential':
        restore_rng(PAPER_INITIAL_RNG)
    records = []
    for case in PROTOCOLS[protocol_name]['cases']:
        print(f'\n[{protocol_name}] {case["target"]} <- {case["reference"]}', flush=True)
        records.append(predict_case(protocol_name, case, run_dir))
    if {k: sha256(v) for k, v in CHECKPOINTS.items()} != identity['checkpoint_sha256']:
        raise RuntimeError('权重文件在运行中发生变化。')
    per_case = pd.DataFrame(records)
    metric_cols = [c for c in per_case if c.startswith(('soft_spearman_', 'soft_f1_'))]
    summary = per_case[metric_cols].agg(['mean', 'std']).T.reset_index(names='metric')
    summary.columns = ['metric', 'mean', 'sample_std']
    summary['n_targets'] = len(records)
    summary['protocol'] = protocol_name
    summary['weight_setting'] = WEIGHT_SETTING
    for name, table in [('per_case.csv', per_case), ('summary.csv', summary)]:
        if (run_dir / name).exists():
            old = pd.read_csv(run_dir / name)
            # 汇总写入后中断时，只接受内容一致的已有文件，不覆盖。
            expected = pd.read_csv(__import__('io').StringIO(table.to_csv(index=False)))
            pd.testing.assert_frame_equal(old, expected)
        else:
            write_csv_new(run_dir / name, table)
    write_json_new(run_dir / 'run_completed.json', {'n_targets': len(records),
                   'files': {name: sha256(run_dir / name) for name in ['per_case.csv', 'summary.csv', 'input_audit.csv']}})
    return per_case, summary, run_dir

## 6. 先运行论文五个固定留出目标

默认 zero-shot，使用源模型直接预测，不在原四张训练切片上重新训练。
遇到 OOM 或其他错误会停止并保留该次 `attempt`；修复运行条件后按第 5 节恢复，或使用新 RUN_ID。

In [ ]:
paper_results, paper_summary, paper_output = run_protocol('paper_rq3')
display(paper_results)
display(paper_summary)
print('论文五个目标已完成：', paper_output)

## 7. 再运行之前的五切片三折规则

使用 `FIVE_SECTIONS` 中五张 Zhuang 切片，内部三张分别作为目标；每折 seed 42，
其余四张可见，左右相邻两张作为参考。不会重新训练模型。
这组是自定义协议，不冒充论文原始划分；与论文组分开汇总。

In [ ]:
threefold_results, threefold_summary, threefold_output = run_protocol('custom_3fold')
display(threefold_results)
display(threefold_summary)
print('三折三个目标已完成：', threefold_output)
assert paper_output != threefold_output
print('两组结果分别保留，未覆盖原有实验目录。')

## 结果目录

```text
results/zhuang_abca2_reconstruction/<weight_setting>/
├── paper_rq3/<RUN_ID>/
│   ├── experiment.json
│   ├── input_audit.csv
│   ├── test_01_sec_021/seed_42/
│   │   ├── attempt_001/  # 每次尝试独立保留
│   │   │   ├── prediction.h5ad
│   │   │   ├── config.json
│   │   │   ├── metrics.json
│   │   │   ├── evaluated_gene_panel.csv
│   │   │   └── rng_after.pt
│   │   └── completed.json  # 成功后才创建
│   ├── ...                 # 共五个测试目标
│   ├── per_case.csv
│   ├── summary.csv
│   └── run_completed.json
└── custom_3fold/<RUN_ID>/    # 相同结构，共三个目标
```

- 默认每次全新 RUN_ID；恢复时仅复用校验一致的完成结果。失败与中断的尝试不会被覆盖。
- 两组都保留 2000 基因预测；直接评估的实测交集单独记录，原指标仍执行自身的表达过滤。
- 预测的 `obsm['spatial']` 保留模型内部 `[z_ccf, y_ccf, x_ccf]`；`spatial_ccf_mm` 为实际 `[x_ccf, y_ccf, z_ccf]`。
- `summary.csv` 的标准差是目标切片间的样本标准差，不是多 seed 方差。
- 原 soft Spearman、soft F1 的半径不变：0.03、0.04、0.05、0.09、0.12、0.15 mm。
- 两个运行单元成功并生成 `run_completed.json` 后，才代表真实重建和评估完成。
- 原始数据、权重、模型源码、之前的下丘脑三折结果及下载/3D 可视化输出均不修改。